## Google Drive Storage

In [28]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    main_path = "/content/drive/MyDrive/AI Models/Tone Detection Model"
except Exception:
    main_path = "model"

In [29]:
print(main_path)

model


## Imports

In [30]:
import pandas as pd
import pickle
import torch
import torch.nn.functional as F 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import BinaryAccuracy
from torchinfo import summary
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import kagglehub
from kagglehub import KaggleDatasetAdapter
import matplotlib.pyplot as plt
import numpy as np

## Dataset

In [31]:
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "devkhant24/toxic-comment",
    "jigsaw-toxic-comment-train.csv"
)
# randomly select 50000 rows
df = df.sample(n=220000, random_state=42)

/tmp/ipykernel_47887/2646130499.py:1: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [32]:
# fillna replaces NaN with ""
df_cleaned = df.dropna(subset=["comment_text"])
texts = df_cleaned["comment_text"].dropna()
y = df_cleaned["toxic"].values

In [33]:
print(texts[:10])

170259    :You might like to consider that I don't give ...
92211     What the heck are you talking about? I ask, my...
102203                              Uncle Tom House Niggers
153827    Well, just because you hate the word doesn't m...
97484                         source: Places named Scotland
194002    " \n\n  Please do not vandalize pages, as you ...
124778    "\n\nVirtually Incomprehensible\n\n""Although ...
39535     "\n\nLike I said, the comments were made by Kh...
65327     Get In the Ring \n\n^^^^^^^^^^^^^^^^^^^^^^^^:)...
40779     Santo DiMera\nYour continued vandalism to the ...
Name: comment_text, dtype: str


In [34]:
print(df[:10])

                      id                                       comment_text  \
170259  2ada3066c863097d  :You might like to consider that I don't give ...   
92211   f685d54247b735a8  What the heck are you talking about? I ask, my...   
102203  22ed5297921900fb                            Uncle Tom House Niggers   
153827  a364a9ef7c480da8  Well, just because you hate the word doesn't m...   
97484   09884c47e5720176                      source: Places named Scotland   
194002  898c44b5f9413c8c  " \n\n  Please do not vandalize pages, as you ...   
124778  9b8e8100ac3195e3  "\n\nVirtually Incomprehensible\n\n""Although ...   
39535   698508c747445410  "\n\nLike I said, the comments were made by Kh...   
65327   aed200739f484219  Get In the Ring \n\n^^^^^^^^^^^^^^^^^^^^^^^^:)...   
40779   6cd44facd1a9f867  Santo DiMera\nYour continued vandalism to the ...   

        toxic  severe_toxic  obscene  threat  insult  identity_hate  
170259      1             0        1       0       1        

In [35]:
print(y[:10])

[1 0 1 0 0 0 0 0 0 0]


In [36]:
# split into words. count frequency of each word, reduce weight for common words. output to numeric array
# Use the same vectoriser when building prediction APIs
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 2),
    min_df=0.03,
    max_df=0.93,
    max_features=10000,
    sublinear_tf=True
)
X = vectorizer.fit_transform(texts).toarray()

In [37]:
print(X[2])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0.]


In [38]:
pickle.dump(vectorizer, open(f"{main_path}/vectorizer.pkl", "wb"))

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [41]:
class ToneDetectionDataset(Dataset):
    def __init__(self, train_feature, train_label):
        self.train_feature = train_feature.astype(np.float32)
        self.train_label = train_label.astype(np.int64)

    def __len__(self):
        return len(self.train_feature)

    def __getitem__(self, idx):
        selected_feature = self.train_feature[idx]
        selected_label = self.train_label[idx]

        selected_feature = torch.tensor(selected_feature, dtype=torch.float32)
        selected_feature = selected_feature.view(1, -1)
        selected_label = torch.tensor(selected_label, dtype=torch.long)

        return selected_label, selected_feature

In [42]:
pt_train_dataset = ToneDetectionDataset(train_feature=X_train, train_label=y_train)
pt_test_dataset = ToneDetectionDataset(train_feature=X_test, train_label=y_test)

In [43]:
sample_label, sample_feature = pt_train_dataset[100]

print(sample_feature.shape)

torch.Size([1, 247])


In [44]:
train_dataloader = DataLoader(pt_train_dataset, batch_size=60, shuffle=True)

## Model

This is a regression model

In [45]:
class ToneDetectionModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=200, stride=1, padding=0, kernel_size=3)
        self.conv2 = nn.Conv1d(in_channels=200, out_channels=150, stride=1, padding=0, kernel_size=3)
        self.conv3 = nn.Conv1d(in_channels=150, out_channels=100, stride=1, padding=0, kernel_size=3)
        self.linear1 = nn.Linear(in_features=100, out_features=64)
        self.linear2 = nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = F.adaptive_avg_pool1d(x, 1)
        x = torch.flatten(x, 1)
        x = F.relu(self.linear1(x))
        x = self.linear2(x).squeeze(1)
        return x

In [46]:
model = ToneDetectionModel()

In [47]:
summary(model, input_size=(60, 1, 247))

Layer (type:depth-idx)                   Output Shape              Param #
ToneDetectionModel                       [60]                      --
├─Conv1d: 1-1                            [60, 200, 245]            800
├─Conv1d: 1-2                            [60, 150, 243]            90,150
├─Conv1d: 1-3                            [60, 100, 241]            45,100
├─Linear: 1-4                            [60, 64]                  6,464
├─Linear: 1-5                            [60, 1]                   65
Total params: 142,579
Trainable params: 142,579
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 1.98
Input size (MB): 0.06
Forward/backward pass size (MB): 52.62
Params size (MB): 0.57
Estimated Total Size (MB): 53.24

## Train

In [48]:
criterion = nn.BCEWithLogitsLoss()
optimiser = optim.AdamW(lr=0.001, params=model.parameters())
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser)


In [49]:
for _ in range(3):
    losses = []

    for batch in train_dataloader:
        label, feature = batch
        label = label.to(torch.float32)

        pred = model(feature)
        loss = criterion(pred, label)

        print(f"{pred.shape} vs {label.shape}")

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        print(f"Loss is : {loss}")

        losses.append(loss.item())

    mean_loss = sum(losses) / len(losses)
    scheduler.step(mean_loss)
    print(f"Epoch {_ + 1} loss: {mean_loss:.4f}")

torch.Size([60]) vs torch.Size([60])
Loss is : 0.6442550420761108
torch.Size([60]) vs torch.Size([60])
Loss is : 0.5399706959724426
torch.Size([60]) vs torch.Size([60])
Loss is : 0.43740159273147583
torch.Size([60]) vs torch.Size([60])
Loss is : 0.26964062452316284
torch.Size([60]) vs torch.Size([60])
Loss is : 0.43635934591293335
torch.Size([60]) vs torch.Size([60])
Loss is : 0.39272111654281616
torch.Size([60]) vs torch.Size([60])
Loss is : 0.36523234844207764
torch.Size([60]) vs torch.Size([60])
Loss is : 0.3123036026954651
torch.Size([60]) vs torch.Size([60])
Loss is : 0.1994611769914627
torch.Size([60]) vs torch.Size([60])
Loss is : 0.2971707880496979
torch.Size([60]) vs torch.Size([60])
Loss is : 0.33455315232276917
torch.Size([60]) vs torch.Size([60])
Loss is : 0.20916512608528137
torch.Size([60]) vs torch.Size([60])
Loss is : 0.21836479008197784
torch.Size([60]) vs torch.Size([60])
Loss is : 0.39555734395980835
torch.Size([60]) vs torch.Size([60])
Loss is : 0.3600122332572937
t

KeyboardInterrupt: 

## Evaluate

- against test data from dataset
    - Loss
    - Accuracy
    - F1
- against custom data
    - Loss
    - Accuracy
    - F1